# Indian Multilingual Intent Detection — Baseline

## Project Objective

This notebook implements the baseline system for intent detection from speech.

### Baseline Pipeline

Audio → Speech-to-Text → Text → Intent Detection → Predicted Intent

In this baseline, the audio signal is first converted into text using a
multilingual Automatic Speech Recognition (ASR) model. The resulting
transcription is then provided to a multilingual text-based intent
classification model.

### Objective of the Baseline

The purpose of this baseline is to establish how well intent detection
can be performed using only the textual information extracted from speech.

A future multimodal extension will incorporate speech representations
alongside textual representations to investigate whether audio-specific
information improves intent detection performance.

### Main Components

1. Audio input
2. Multilingual Speech-to-Text
3. Text preprocessing
4. Multilingual Intent Classification
5. Intent prediction
6. Model evaluation

In [ ]:
# ============================================================
# Cell 2: Environment Verification
# Purpose: Verify the Python environment and available hardware
# ============================================================

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available: Yes")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU available: No")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cu128
GPU available: Yes
GPU: Tesla T4


In [ ]:
# ============================================================
# Cell 3: Install Baseline Dependencies
# Purpose: Install libraries required for multilingual
#          speech-to-text and audio processing
# ============================================================

!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# Cell 4: Import Required Libraries
# Purpose: Import libraries required for the baseline pipeline
# ============================================================

import whisper
import torch

print("Whisper imported successfully.")
print("CUDA available:", torch.cuda.is_available())

Whisper imported successfully.
CUDA available: True


In [ ]:
# ============================================================
# Cell 5: Load Whisper Speech-to-Text Model
# Purpose: Load a pretrained multilingual ASR model
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

whisper_model = whisper.load_model("small", device=DEVICE)

print("Whisper model loaded successfully.")

Using device: cuda


100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 75.3MiB/s]


Whisper model loaded successfully.


In [ ]:
# ============================================================
# Cell 6: Upload Test Audio
# Purpose: Upload one audio sample for testing the
#          speech-to-text pipeline
# ============================================================

from google.colab import files

uploaded = files.upload()

audio_file = next(iter(uploaded))

print("Uploaded audio file:", audio_file)

Saving test1.mp4 to test1.mp4
Uploaded audio file: test1.mp4


In [ ]:
# ============================================================
# Cell 7: Speech-to-Text Transcription
# Purpose: Convert the uploaded speech into text using Whisper
# ============================================================

result = whisper_model.transcribe(
    audio_file,
    task="transcribe"
)

transcribed_text = result["text"].strip()
detected_language = result["language"]

print("Detected language:", detected_language)
print("Transcribed text:", transcribed_text)

Detected language: hi
Transcribed text: मुछ्टे भूग लग्र रही है


In [ ]:
# ============================================================
# Cell 8: Load Whisper Large Model
# Purpose: Compare a larger Whisper model with the Small model
#          for Indian-language speech recognition
# ============================================================

whisper_medium = whisper.load_model("medium", device=DEVICE)

print("Whisper Large model loaded successfully.")

100%|█████████████████████████████████████| 2.88G/2.88G [00:43<00:00, 70.5MiB/s]


Whisper Large model loaded successfully.


In [ ]:
# ============================================================
# Cell 9: Telugu Speech Transcription with Whisper Medium
# Purpose: Evaluate whether the larger model improves
#          transcription quality for Telugu speech
# ============================================================

medium_result = whisper_medium.transcribe(
    audio_file,
    task="transcribe"
)

medium_text = medium_result["text"].strip()
medium_language = medium_result["language"]

print("Detected language:", medium_language)
print("Transcribed text:", medium_text)

Detected language: hi
Transcribed text: मुझे भूक लग रही है


In [ ]:
# ============================================================
# Cell 12: Load MASSIVE Telugu Dataset
# Purpose: Load the Telugu portion of MASSIVE directly from
#          the verified Parquet-conversion revision
# ============================================================

from datasets import load_dataset

massive_te = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/te-IN/train/0000.parquet",
        "validation": "https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/te-IN/validation/0000.parquet",
        "test": "https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/te-IN/test/0000.parquet"
    }
)

print("Telugu MASSIVE dataset loaded successfully.")
print(massive_te)

te-IN/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.18MB            

te-IN/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/231k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/322k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Telugu MASSIVE dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})


In [ ]:
# ============================================================
# Cell 13: Inspect Telugu Intent Data
# Purpose: Examine utterances, intent labels, and their
#          distribution in the Telugu MASSIVE dataset
# ============================================================

train_data = massive_te["train"]

print("Number of training examples:", len(train_data))

# Show the first few examples
print("\nSample Telugu utterances:")
for i in range(5):
    print(f"{i + 1}. Text: {train_data[i]['utt']}")
    print(f"   Intent: {train_data[i]['intent']}")
    print()

# Count unique intents
intent_counts = {}

for example in train_data:
    intent = example["intent"]
    intent_counts[intent] = intent_counts.get(intent, 0) + 1

print("Number of unique intents:", len(intent_counts))

print("\nIntent distribution:")
for intent, count in sorted(intent_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{intent}: {count}")

Number of training examples: 11514

Sample Telugu utterances:
1. Text: శుక్రవారం ఉదయం తొమ్మిది గంటలు కి నన్ను నిద్ర లేపు
   Intent: 48

2. Text: ఇప్పటి నుండి రెండు గంటలకు అలారం సెట్ చేయి
   Intent: 48

3. Text: ఒల్లీ నిశ్శబ్దంగా ఉండు
   Intent: 46

4. Text: ఆపండి
   Intent: 46

5. Text: ఓలి పదిహేను సెకండ్లు పోజ్ చేయి
   Intent: 46

Number of unique intents: 60

Intent distribution:
50: 810
45: 639
13: 573
32: 566
12: 555
49: 544
22: 503
44: 418
33: 354
0: 350
30: 312
36: 283
47: 283
26: 267
42: 227
9: 207
59: 198
58: 193
6: 190
48: 182
21: 177
19: 173
53: 164
57: 154
40: 153
4: 152
20: 150
10: 142
16: 135
23: 130
2: 127
17: 127
1: 125
56: 124
3: 122
11: 117
43: 113
51: 112
46: 110
14: 110
27: 108
54: 100
34: 93
52: 78
39: 78
31: 76
18: 76
25: 72
55: 70
15: 54
38: 52
8: 52
35: 52
28: 51
24: 48
5: 25
41: 22
29: 18
7: 14
37: 4


In [ ]:
# ============================================================
# Cell 14: Decode Intent Labels
# Purpose: Map numerical intent IDs to their human-readable
#          intent names
# ============================================================

print("Dataset features:")
print(train_data.features)

print("\nIntent feature:")
print(train_data.features["intent"])

Dataset features:
{'id': Value('string'), 'locale': Value('string'), 'partition': Value('string'), 'scenario': ClassLabel(names=['social', 'transport', 'calendar', 'play', 'news', 'datetime', 'recommendation', 'email', 'iot', 'general', 'audio', 'lists', 'qa', 'cooking', 'takeaway', 'music', 'alarm', 'weather']), 'intent': ClassLabel(names=['datetime_query', 'iot_hue_lightchange', 'transport_ticket', 'takeaway_query', 'qa_stock', 'general_greet', 'recommendation_events', 'music_dislikeness', 'iot_wemo_off', 'cooking_recipe', 'qa_currency', 'transport_traffic', 'general_quirky', 'weather_query', 'audio_volume_up', 'email_addcontact', 'takeaway_order', 'email_querycontact', 'iot_hue_lightup', 'recommendation_locations', 'play_audiobook', 'lists_createoradd', 'news_query', 'alarm_query', 'iot_wemo_on', 'general_joke', 'qa_definition', 'social_query', 'music_settings', 'audio_volume_other', 'calendar_remove', 'iot_hue_lightdim', 'calendar_query', 'email_sendemail', 'iot_cleaning', 'audio_v

In [ ]:
# ============================================================
# Cell 15: Check MASSIVE Language Configurations
# Purpose: Identify the available language configurations
#          for constructing our Indian-language dataset
# ============================================================

from huggingface_hub import list_repo_tree

repo_id = "AmazonScience/massive"

print("Available MASSIVE language configurations:\n")

items = list_repo_tree(
    repo_id,
    repo_type="dataset",
    recursive=False
)

for item in items:
    print(item.path)

Available MASSIVE language configurations:

.gitattributes
CITATION.cff
LICENSE
README.md
massive.py
test_massive.py


In [ ]:
# ============================================================
# Cell 16: Load Remaining Indian-Language MASSIVE Datasets
# Purpose: Load Hindi, Kannada, Malayalam, and Tamil data
#          for the multilingual Indian-language baseline
# ============================================================

from datasets import load_dataset

indian_locales = {
    "hi": "hi-IN",
    "kn": "kn-IN",
    "ml": "ml-IN",
    "ta": "ta-IN"
}

massive_indian = {}

for language, locale in indian_locales.items():

    print(f"\nLoading {locale}...")

    dataset = load_dataset(
        "parquet",
        data_files={
            "train": f"https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/{locale}/train/0000.parquet",
            "validation": f"https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/{locale}/validation/0000.parquet",
            "test": f"https://huggingface.co/datasets/AmazonScience/massive/resolve/refs%2Fconvert%2Fparquet/{locale}/test/0000.parquet"
        }
    )

    massive_indian[language] = dataset

    print(f"{locale} loaded successfully.")
    print(dataset)


Loading hi-IN...


hi-IN/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.10MB            

hi-IN/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/215k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/299k [00:00<?, ?B/s]

...

0000.parquet:   0%|          | 0.00/326k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

ta-IN loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})


In [ ]:
# ============================================================
# Cell 17: Verify Intent Label Consistency
# Purpose: Confirm that all five Indian-language datasets
#          use the same intent ID → intent name mapping
# ============================================================

languages = {
    "Hindi": massive_indian["hi"],
    "Kannada": massive_indian["kn"],
    "Malayalam": massive_indian["ml"],
    "Tamil": massive_indian["ta"],
    "Telugu": massive_te
}

reference_names = None
mapping_consistent = True

for language, dataset in languages.items():

    intent_names = dataset["train"].features["intent"].names

    print(f"{language}: {len(intent_names)} intents")

    if reference_names is None:
        reference_names = intent_names
    elif intent_names != reference_names:
        mapping_consistent = False
        print(f"WARNING: Intent mapping differs for {language}")

print("\nIntent mapping consistent across all languages:",
      mapping_consistent)

if mapping_consistent:
    print("\nVerified intent labels:")
    for index, name in enumerate(reference_names):
        print(f"{index:2d} → {name}")

Hindi: 60 intents
Kannada: 60 intents
Malayalam: 60 intents
Tamil: 60 intents
Telugu: 60 intents

Intent mapping consistent across all languages: True

Verified intent labels:
 0 → datetime_query
 1 → iot_hue_lightchange
 2 → transport_ticket
 3 → takeaway_query
 4 → qa_stock
 5 → general_greet
 6 → recommendation_events
 7 → music_dislikeness
 8 → iot_wemo_off
 9 → cooking_recipe
10 → qa_currency
11 → transport_traffic
12 → general_quirky
13 → weather_query
14 → audio_volume_up
15 → email_addcontact
16 → takeaway_order
17 → email_querycontact
18 → iot_hue_lightup
19 → recommendation_locations
20 → play_audiobook
21 → lists_createoradd
22 → news_query
23 → alarm_query
24 → iot_wemo_on
25 → general_joke
26 → qa_definition
27 → social_query
28 → music_settings
29 → audio_volume_other
30 → calendar_remove
31 → iot_hue_lightdim
32 → calendar_query
33 → email_sendemail
34 → iot_cleaning
35 → audio_volume_down
36 → play_radio
37 → cooking_query
38 → datetime_convert
39 → qa_maths
40 → iot_hu

In [ ]:
# ============================================================
# Cell 18: Combine Indian-Language MASSIVE Datasets
# Purpose: Create unified train, validation, and test sets
#          containing Hindi, Kannada, Malayalam, Tamil,
#          and Telugu
# ============================================================

from datasets import concatenate_datasets, DatasetDict

# Collect datasets in a fixed language order
language_datasets = [
    massive_indian["hi"],
    massive_indian["kn"],
    massive_indian["ml"],
    massive_indian["ta"],
    massive_te
]

# Combine each split separately
combined_train = concatenate_datasets(
    [dataset["train"] for dataset in language_datasets]
)

combined_validation = concatenate_datasets(
    [dataset["validation"] for dataset in language_datasets]
)

combined_test = concatenate_datasets(
    [dataset["test"] for dataset in language_datasets]
)

# Create the final multilingual dataset
massive_indian_combined = DatasetDict({
    "train": combined_train,
    "validation": combined_validation,
    "test": combined_test
})

print("Combined Indian-language MASSIVE dataset created.")

print("\nDataset sizes:")
print("Train:", len(massive_indian_combined["train"]))
print("Validation:", len(massive_indian_combined["validation"]))
print("Test:", len(massive_indian_combined["test"]))

Combined Indian-language MASSIVE dataset created.

Dataset sizes:
Train: 57570
Validation: 10165
Test: 14870


In [ ]:
# ============================================================
# Cell 19: Verify Combined Language Distribution
# Purpose: Confirm balanced representation of the five
#          Indian languages in the combined dataset
# ============================================================

from collections import Counter

train_data = massive_indian_combined["train"]

# Count examples by language
language_counts = Counter(train_data["locale"])

print("Training examples by language:\n")

for language, count in sorted(language_counts.items()):
    print(f"{language}: {count}")

print("\nTotal training examples:", len(train_data))

# Show one example from each language
print("\nSample utterance from each language:\n")

shown_languages = set()

for example in train_data:

    language = example["locale"]

    if language not in shown_languages:
        print(f"Language: {language}")
        print(f"Text: {example['utt']}")
        print(f"Intent: {example['intent']}")
        print()

        shown_languages.add(language)

    if len(shown_languages) == 5:
        break

Training examples by language:

hi-IN: 11514
kn-IN: 11514
ml-IN: 11514
ta-IN: 11514
te-IN: 11514

Total training examples: 57570

Sample utterance from each language:

Language: hi-IN
Text: शुक्रवार को सुबह नौ बजे मुझे जगा दो
Intent: 48

Language: kn-IN
Text: ಶುಕ್ರವಾರ ಬೆಳಗ್ಗೆ ಒಂಬತ್ತು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎಬ್ಬಿಸಿ ಗುರಿ ಅಲಾರಂ ಹೊಂದಿಸುವ ಕುರಿತು ನಿಮ್ಮ ವೈಯಕ್ತಿಕ ಸಹಾಯಕರನ್ನು ಕೇಳಲಾಗುತ್ತಿದೆ
Intent: 48

Language: ml-IN
Text: വെള്ളിയാഴ്ച രാവിലെ ഒമ്പത് മണിക്ക് എന്നെ ഉണർത്തുക
Intent: 48

Language: ta-IN
Text: வெள்ளிக்கிழமை காலை ஒன்பது மணிக்கு என்னை எழுப்புங்கள்
Intent: 48

Language: te-IN
Text: శుక్రవారం ఉదయం తొమ్మిది గంటలు కి నన్ను నిద్ర లేపు
Intent: 48



In [ ]:
# ============================================================
# Cell 20: Install Transformer Dependencies
# Purpose: Install the Hugging Face Transformers library
#          required for loading and fine-tuning MuRIL
# ============================================================

!pip install -q transformers accelerate

In [ ]:
# ============================================================
# Cell 21: Load Pretrained MuRIL
# Purpose: Load the pretrained MuRIL tokenizer and encoder
#          that will be fine-tuned for intent classification
# ============================================================

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
muril = AutoModel.from_pretrained(MODEL_NAME)

print("MuRIL loaded successfully.")
print("Model:", MODEL_NAME)
print("Hidden size:", muril.config.hidden_size)
print("Number of layers:", muril.config.num_hidden_layers)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRIL loaded successfully.
Model: google/muril-base-cased
Hidden size: 768
Number of layers: 12


In [ ]:
# ============================================================
# Cell 22: Test MuRIL Tokenization
# Purpose: Verify that the MuRIL tokenizer correctly processes
#          text from all five Indian languages
# ============================================================

sample_texts = {
    "Hindi": "शुक्रवार को सुबह नौ बजे मुझे जगा दो",
    "Kannada": "ಶುಕ್ರವಾರ ಬೆಳಗ್ಗೆ ಒಂಬತ್ತು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎಬ್ಬಿಸಿ",
    "Malayalam": "വെള്ളിയാഴ്ച രാവിലെ ഒമ്പത് മണിക്ക് എന്നെ ഉണർത്തുക",
    "Tamil": "வெள்ளிக்கிழமை காலை ஒன்பது மணிக்கு என்னை எழுப்புங்கள்",
    "Telugu": "శుక్రవారం ఉదయం తొమ్మిది గంటలు కి నన్ను నిద్ర లేపు"
}

for language, text in sample_texts.items():

    encoded = tokenizer(
        text,
        return_tensors="pt"
    )

    print(f"{language}:")
    print("Text:", text)
    print("Token IDs:", encoded["input_ids"].shape)
    print()

Hindi:
Text: शुक्रवार को सुबह नौ बजे मुझे जगा दो
Token IDs: torch.Size([1, 10])

Kannada:
Text: ಶುಕ್ರವಾರ ಬೆಳಗ್ಗೆ ಒಂಬತ್ತು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎಬ್ಬಿಸಿ
Token IDs: torch.Size([1, 10])

Malayalam:
Text: വെള്ളിയാഴ്ച രാവിലെ ഒമ്പത് മണിക്ക് എന്നെ ഉണർത്തുക
Token IDs: torch.Size([1, 10])

Tamil:
Text: வெள்ளிக்கிழமை காலை ஒன்பது மணிக்கு என்னை எழுப்புங்கள்
Token IDs: torch.Size([1, 10])

Telugu:
Text: శుక్రవారం ఉదయం తొమ్మిది గంటలు కి నన్ను నిద్ర లేపు
Token IDs: torch.Size([1, 11])



In [ ]:
# ============================================================
# Cell 23: Prepare Dataset for MuRIL Fine-Tuning
# Purpose: Keep only the text, intent label, and language
#          needed for multilingual intent classification
# ============================================================

def prepare_for_muril(example):
    return {
        "text": example["utt"],
        "labels": example["intent"]
    }

muril_dataset = massive_indian_combined.map(
    prepare_for_muril,
    remove_columns=massive_indian_combined["train"].column_names
)

print("Dataset prepared for MuRIL.")

print("\nFeatures:")
print(muril_dataset["train"].features)

print("\nExample:")
print(muril_dataset["train"][0])

Map:   0%|          | 0/57570 [00:00<?, ? examples/s]

Map:   0%|          | 0/10165 [00:00<?, ? examples/s]

Map:   0%|          | 0/14870 [00:00<?, ? examples/s]

Dataset prepared for MuRIL.

Features:
{'text': Value('string'), 'labels': Value('int64')}

Example:
{'text': 'शुक्रवार को सुबह नौ बजे मुझे जगा दो', 'labels': 48}


In [ ]:
# ============================================================
# Cell 24: Tokenize MASSIVE Dataset
# Purpose: Convert multilingual Indian-language text into
#          numerical token IDs that MuRIL can process
# ============================================================

MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding=False,
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = muril_dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully.")

print("\nFeatures:")
print(tokenized_dataset["train"].features)

print("\nSample tokenized example:")
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/57570 [00:00<?, ? examples/s]

Map:   0%|          | 0/10165 [00:00<?, ? examples/s]

Map:   0%|          | 0/14870 [00:00<?, ? examples/s]

Dataset tokenized successfully.

Features:
{'text': Value('string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}

Sample tokenized example:
{'text': 'शुक्रवार को सुबह नौ बजे मुझे जगा दो', 'labels': 48, 'input_ids': [104, 14955, 1125, 8814, 13011, 6353, 4254, 162523, 1940, 105], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
# ============================================================
# Cell 25: Create MuRIL Intent Classifier
# Purpose: Add a 60-class intent classification head on top
#          of the pretrained MuRIL encoder
# ============================================================

from transformers import AutoModelForSequenceClassification

NUM_LABELS = 60

id2label = {
    i: name
    for i, name in enumerate(
        massive_te["train"].features["intent"].names
    )
}

label2id = {
    name: i
    for i, name in id2label.items()
}

muril_classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

print("MuRIL intent classifier created successfully.")
print("Number of labels:", muril_classifier.config.num_labels)
print("Hidden size:", muril_classifier.config.hidden_size)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

MuRIL intent classifier created successfully.
Number of labels: 60
Hidden size: 768


In [ ]:
# ============================================================
# Cell 26: Configure MuRIL Fine-Tuning
# Purpose: Define the training and evaluation settings for
#          our multilingual Indian-language intent baseline
# ============================================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./muril_indian_intent_baseline",

    # Training
    num_train_epochs=5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,

    # Evaluation
    per_device_eval_batch_size=32,
    eval_strategy="epoch",

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,

    # Performance
    fp16=True,

    # Save the best model
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # Logging
    logging_strategy="steps",
    logging_steps=100,

    # Reproducibility
    seed=42,

    # Disable external experiment tracking
    report_to="none"
)

print("Training configuration created successfully.")
print(training_args)

Training configuration created successfully.
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=

In [ ]:
# ============================================================
# Cell 27: Define Evaluation Metrics
# Purpose: Measure MuRIL's performance using accuracy,
#          macro F1, and weighted F1
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_prediction):

    predictions, labels = eval_prediction

    # Convert model outputs into predicted intent IDs
    predictions = np.argmax(predictions, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

print("Evaluation metrics defined:")
print("- Accuracy")
print("- Macro F1")
print("- Weighted F1")

Evaluation metrics defined:
- Accuracy
- Macro F1
- Weighted F1


In [ ]:
# ============================================================
# Cell 28: Create Dynamic Data Collator
# Purpose: Dynamically pad each training batch so that
#          sentences of different lengths can be processed
#          efficiently by MuRIL
# ============================================================

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

print("Dynamic data collator created successfully.")

Dynamic data collator created successfully.


In [ ]:
# ============================================================
# Cell 29: Confirm GPU Availability
# Purpose: Verify that the Colab runtime has reconnected to
#          the Tesla T4 before starting MuRIL fine-tuning
# ============================================================

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2
    ), "GB")
else:
    print("WARNING: GPU is not available.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# ============================================================
# Cell 30: Create MuRIL Trainer
# Purpose: Connect the model, tokenized datasets, data
#          collator, training configuration, and metrics
# ============================================================

from transformers import Trainer

trainer = Trainer(
    model=muril_classifier,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("MuRIL Trainer created successfully.")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Number of intents:", NUM_LABELS)

NameError: name 'muril_classifier' is not defined

In [ ]:
# ============================================================
# Cell 31: Fine-Tune MuRIL for Indian-Language Intent Detection
# Purpose: Train MuRIL on the five-language MASSIVE dataset
#          containing 60 intent categories
# ============================================================

print("Starting MuRIL fine-tuning...")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Languages: Hindi, Kannada, Malayalam, Tamil, Telugu")
print("Number of intents:", NUM_LABELS)
print()

training_result = trainer.train()

print("\nMuRIL fine-tuning completed successfully.")

Starting MuRIL fine-tuning...
Training examples: 57570
Validation examples: 10165
Languages: Hindi, Kannada, Malayalam, Tamil, Telugu
Number of intents: 60



Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.430913,1.420615,0.666503,0.376702,0.611556
2,0.688475,0.882872,0.814166,0.664819,0.800183
3,0.418901,0.805589,0.840728,0.750420,0.835014
4,0.295597,0.830837,0.852730,0.787870,0.848400
5,0.176306,0.855022,0.856272,0.795287,0.852775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MuRIL fine-tuning completed successfully.


In [ ]:
# ============================================================
# Cell 32: Evaluate MuRIL on the Unseen Test Set
# Purpose: Measure final baseline performance on the held-out
#          test set that was not used during training
# ============================================================

print("Evaluating MuRIL on the unseen test set...")

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nFinal Test Results:")
print("----------------------------------------")

for metric, value in test_results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

Evaluating MuRIL on the unseen test set...


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.176306,0.814164,5,0.860659,0.795035,0.858768



Final Test Results:
----------------------------------------
eval_loss: 0.8142
eval_accuracy: 0.8607
eval_macro_f1: 0.7950
eval_weighted_f1: 0.8588


In [ ]:
# ============================================================
# Cell 33: Evaluate MuRIL Performance by Language
# Purpose: Measure intent classification performance separately
#          for Hindi, Kannada, Malayalam, Tamil, and Telugu
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Get predictions on the complete test set
predictions_output = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = np.array(
    tokenized_dataset["test"]["labels"]
)

# Get language information from the original combined dataset
test_locales = np.array(
    massive_indian_combined["test"]["locale"]
)

print("Per-language test performance")
print("=" * 60)

for locale in sorted(np.unique(test_locales)):

    mask = test_locales == locale

    language_accuracy = accuracy_score(
        true_labels[mask],
        predicted_labels[mask]
    )

    language_macro_f1 = f1_score(
        true_labels[mask],
        predicted_labels[mask],
        average="macro",
        zero_division=0
    )

    language_weighted_f1 = f1_score(
        true_labels[mask],
        predicted_labels[mask],
        average="weighted",
        zero_division=0
    )

    print(f"\n{locale}")
    print(f"Examples:      {mask.sum()}")
    print(f"Accuracy:      {language_accuracy:.4f}")
    print(f"Macro F1:      {language_macro_f1:.4f}")
    print(f"Weighted F1:   {language_weighted_f1:.4f}")

Per-language test performance

hi-IN
Examples:      2974
Accuracy:      0.8726
Macro F1:      0.8035
Weighted F1:   0.8700

kn-IN
Examples:      2974
Accuracy:      0.8568
Macro F1:      0.7972
Weighted F1:   0.8550

ml-IN
Examples:      2974
Accuracy:      0.8625
Macro F1:      0.7943
Weighted F1:   0.8610

ta-IN
Examples:      2974
Accuracy:      0.8561
Macro F1:      0.7856
Weighted F1:   0.8543

te-IN
Examples:      2974
Accuracy:      0.8554
Macro F1:      0.7913
Weighted F1:   0.8533


In [ ]:
# ============================================================
# Cell 34: Analyze Performance by Intent
# Purpose: Identify which of the 60 intent categories are
#          easiest and hardest for the text-only baseline
# ============================================================

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# Calculate per-intent metrics
precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    labels=list(range(NUM_LABELS)),
    zero_division=0
)

intent_names = [
    id2label[i]
    for i in range(NUM_LABELS)
]

intent_results = pd.DataFrame({
    "intent_id": range(NUM_LABELS),
    "intent": intent_names,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "test_examples": support
})

# Sort by F1 so the hardest intents appear first
hardest_intents = intent_results.sort_values(
    "f1",
    ascending=True
)

print("10 Hardest Intents")
print("=" * 80)

print(
    hardest_intents.head(10).to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format
        }
    )
)

print("\n\n10 Best Performing Intents")
print("=" * 80)

best_intents = intent_results.sort_values(
    "f1",
    ascending=False
)

print(
    best_intents.head(10).to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format
        }
    )
)

10 Hardest Intents
 intent_id             intent precision recall     f1  test_examples
         5      general_greet    0.0000 0.0000 0.0000              5
        29 audio_volume_other    0.0000 0.0000 0.0000             30
        41    iot_hue_lighton    0.0000 0.0000 0.0000             15
        37      cooking_query    0.0000 0.0000 0.0000              0
        28     music_settings    0.4444 0.2667 0.3333             30
         7  music_dislikeness    1.0000 0.2000 0.3333             20
        14    audio_volume_up    0.5106 0.7385 0.6038             65
        12     general_quirky    0.7025 0.5728 0.6310            845
        15   email_addcontact    0.5949 0.7833 0.6763             60
        17 email_querycontact    0.5988 0.7923 0.6821            130


10 Best Performing Intents
 intent_id           intent precision recall     f1  test_examples
        52     alarm_remove    0.9369 0.9905 0.9630            105
         2 transport_ticket    0.9708 0.9486 0.9595        

In [ ]:
# ============================================================
# Cell 35: Analyze Intent Confusions
# Purpose: Identify which intents the text-only MuRIL baseline
#          most frequently confuses with one another
# ============================================================

from sklearn.metrics import confusion_matrix

# Create confusion matrix
cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=list(range(NUM_LABELS))
)

# Find the most common incorrect predictions
confusions = []

for true_id in range(NUM_LABELS):
    for predicted_id in range(NUM_LABELS):

        if true_id != predicted_id and cm[true_id, predicted_id] > 0:

            confusions.append({
                "true_intent": id2label[true_id],
                "predicted_intent": id2label[predicted_id],
                "examples": cm[true_id, predicted_id]
            })

confusions_df = pd.DataFrame(confusions)

confusions_df = confusions_df.sort_values(
    "examples",
    ascending=False
)

print("Top 20 Most Common Intent Confusions")
print("=" * 90)

print(
    confusions_df.head(20).to_string(index=False)
)

Top 20 Most Common Intent Confusions
          true_intent      predicted_intent  examples
       general_quirky            qa_factoid        65
       general_quirky        calendar_query        59
           qa_factoid        general_quirky        50
         calendar_set        calendar_query        49
       general_quirky            news_query        46
           play_music        play_audiobook        25
       calendar_query recommendation_events        23
   audio_volume_other       audio_volume_up        23
        qa_definition        general_quirky        23
           qa_factoid    email_querycontact        22
           play_radio            play_music        22
         lists_remove       calendar_remove        21
recommendation_events        calendar_query        21
           news_query              qa_stock        20
      email_sendemail      email_addcontact        19
       general_quirky         weather_query        19
           news_query        general_quirky  

In [ ]:
# ============================================================
# Cell 36: Inspect Misclassified Examples
# Purpose: Examine actual test utterances behind common
#          intent confusions
# ============================================================

# Keep only the original text and labels from the test set
test_texts = massive_indian_combined["test"]["utt"]

# Select important confusion pairs
confusion_pairs = [
    ("iot_hue_lightup", "iot_hue_lightoff"),
    ("alarm_query", "alarm_set"),
    ("play_audiobook", "play_podcasts"),
    ("transport_taxi", "transport_ticket")
]

print("Inspecting selected intent confusions")
print("=" * 100)

for true_intent, predicted_intent in confusion_pairs:

    true_id = label2id[true_intent]
    predicted_id = label2id[predicted_intent]

    print(f"\nTRUE: {true_intent}")
    print(f"PREDICTED: {predicted_intent}")
    print("-" * 100)

    count = 0

    for i in range(len(true_labels)):

        if (
            true_labels[i] == true_id
            and predicted_labels[i] == predicted_id
        ):

            print(
                f"Example {count + 1}: "
                f"{test_locales[i]} → {test_texts[i]}"
            )

            count += 1

            if count >= 5:
                break

Inspecting selected intent confusions

TRUE: iot_hue_lightup
PREDICTED: iot_hue_lightoff
----------------------------------------------------------------------------------------------------
Example 1: hi-IN → मैं बत्तियां खुली नहीं देख सकता
Example 2: kn-IN → ನಾನು ದೀಪಗಳನ್ನು ಆನ್ ಮಾಡುವುದನ್ನು ನೋಡಲು ಸಾಧ್ಯವಿಲ್ಲ
Example 3: ml-IN → മുകളിലേക്ക് വിളക്കുകൾ ഓഫ് ആക്കുക
Example 4: te-IN → నేను లైట్లు వెలిగించడం చూడలేను

TRUE: alarm_query
PREDICTED: alarm_set
----------------------------------------------------------------------------------------------------
Example 1: hi-IN → मेरा सेट अलार्म
Example 2: hi-IN → अलार्म सेटिंग
Example 3: hi-IN → क्या मुझे अपने अगले डॉक्टर के नियुक्ति के लिए रिमाइंडर अलार्म सेट करना याद है
Example 4: hi-IN → नृत्य प्रतियोगिता के लिए मेरा रिमाइंडर अलार्म सेट
Example 5: hi-IN → दोपहर में टैबलेट रखने के लिए अलार्म सेट करें

TRUE: play_audiobook
PREDICTED: play_podcasts
----------------------------------------------------------------------------------------------------
Exa

In [ ]:
import gc
import torch

# Delete model variables held in memory
for var in ['whisper_model', 'whisper_medium']:
    if var in globals():
        del globals()[var]

# Force garbage collector and release CUDA memory cache
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared successfully!")

GPU memory cleared successfully!


In [ ]:
# ============================================================
# Cell 37: Load Whisper for Speech-to-Text
# Purpose: Prepare Whisper to convert Indian-language speech
#          into text before passing it to MuRIL
# ============================================================

import whisper
import torch

print("Loading Whisper...")

whisper_model = whisper.load_model(
    "large-v3",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Whisper loaded successfully.")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Loading Whisper...
Whisper loaded successfully.
Device: cuda


In [ ]:
# ============================================================
# Cell 38: Upload Test Audio
# Purpose: Upload a short Indian-language speech recording
#          for Whisper transcription
# ============================================================

from google.colab import files

uploaded = files.upload()

audio_path = next(iter(uploaded.keys()))

print("Uploaded audio file:", audio_path)

Saving test1.mp4 to test1 (2).mp4
Uploaded audio file: test1 (2).mp4


In [ ]:
# ============================================================
# Cell 39: Transcribe Test Audio with Whisper
# Purpose: Convert the Indian-language speech recording into
#          text and identify the detected language
# ============================================================

result = whisper_model.transcribe(
    audio_path,
    task="transcribe"
)

detected_language = result["language"]
transcribed_text = result["text"].strip()

print("Detected language:", detected_language)
print("Transcribed text:", transcribed_text)

Detected language: hi
Transcribed text: मुझे भूक लग रही है


In [ ]:
# ============================================================
# Cell 40: Predict Intent from Whisper Transcription
# Purpose: Pass Whisper's transcription into the trained
#          MuRIL intent classifier
# ============================================================

text = transcribed_text

# Tokenize the Whisper transcription
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

# Automatically use the same device as the trained classifier
model_device = next(muril_classifier.parameters()).device

# Move inputs to the classifier's device
inputs = {
    key: value.to(model_device)
    for key, value in inputs.items()
}

# Run the trained intent classifier
with torch.no_grad():
    outputs = muril_classifier(**inputs)

# Get predicted class
predicted_class_id = torch.argmax(outputs.logits, dim=1).item()

print("Detected language:", detected_language)
print("Whisper transcription:", transcribed_text)
print("Predicted class ID:", predicted_class_id)

NameError: name 'tokenizer' is not defined

In [ ]:
# ============================================================
# Cell 41: Convert Predicted Class ID to Intent Name
# Purpose: Map the predicted class ID back to the actual
#          MASSIVE intent label
# ============================================================

print("Predicted class ID:", predicted_class_id)

# Check the classifier's label mapping
print("\nModel label mapping:")
print(muril_classifier.config.id2label)

# Get the predicted intent name
predicted_intent = muril_classifier.config.id2label[predicted_class_id]

print("\nPredicted intent:", predicted_intent)

Predicted class ID: 12

Model label mapping:
{0: 'datetime_query', 1: 'iot_hue_lightchange', 2: 'transport_ticket', 3: 'takeaway_query', 4: 'qa_stock', 5: 'general_greet', 6: 'recommendation_events', 7: 'music_dislikeness', 8: 'iot_wemo_off', 9: 'cooking_recipe', 10: 'qa_currency', 11: 'transport_traffic', 12: 'general_quirky', 13: 'weather_query', 14: 'audio_volume_up', 15: 'email_addcontact', 16: 'takeaway_order', 17: 'email_querycontact', 18: 'iot_hue_lightup', 19: 'recommendation_locations', 20: 'play_audiobook', 21: 'lists_createoradd', 22: 'news_query', 23: 'alarm_query', 24: 'iot_wemo_on', 25: 'general_joke', 26: 'qa_definition', 27: 'social_query', 28: 'music_settings', 29: 'audio_volume_other', 30: 'calendar_remove', 31: 'iot_hue_lightdim', 32: 'calendar_query', 33: 'email_sendemail', 34: 'iot_cleaning', 35: 'audio_volume_down', 36: 'play_radio', 37: 'cooking_query', 38: 'datetime_convert', 39: 'qa_maths', 40: 'iot_hue_lightoff', 41: 'iot_hue_lighton', 42: 'transport_query', 4

In [ ]:
# ============================================================
# Cell 42: Show Top-5 Intent Predictions
# Purpose: See which intents MuRIL considered most likely
#          for the Whisper-generated transcription
# ============================================================

# Get probabilities for all 60 intents
probabilities = torch.softmax(outputs.logits, dim=1)[0]

# Get the top 5 predictions
top5_probabilities, top5_class_ids = torch.topk(probabilities, 5)

print("Whisper transcription:", transcribed_text)
print("\nTop 5 predicted intents:\n")

for rank, (class_id, probability) in enumerate(
    zip(top5_class_ids, top5_probabilities), start=1
):
    intent_name = muril_classifier.config.id2label[class_id.item()]

    print(
        f"{rank}. {intent_name} "
        f"→ {probability.item() * 100:.2f}%"
    )

Whisper transcription: मुझे बूक लग रही है।

Top 5 predicted intents:

1. general_quirky → 80.37%
2. social_query → 3.72%
3. general_greet → 3.56%
4. general_joke → 2.52%
5. social_post → 1.29%


In [ ]:
# ============================================================
# Cell 43: Clean Hindi Text Intent Test
# Purpose: Check MuRIL directly on the correct Hindi sentence
#          without Whisper
# ============================================================

clean_text = "मेरे लिए ऑनलाइन सर्वेक्षणों के बारे में नवीनतम पॉडकास्ट चालू करें"

# Tokenize the clean text
clean_inputs = tokenizer(
    clean_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

# Use the same device as the trained classifier
model_device = next(muril_classifier.parameters()).device

clean_inputs = {
    key: value.to(model_device)
    for key, value in clean_inputs.items()
}

# Predict
with torch.no_grad():
    clean_outputs = muril_classifier(**clean_inputs)

# Probabilities
clean_probabilities = torch.softmax(
    clean_outputs.logits, dim=1
)[0]

# Top 5
top5_probs, top5_ids = torch.topk(clean_probabilities, 5)

print("Clean Hindi text:", clean_text)
print("\nTop 5 predicted intents:\n")

for rank, (class_id, probability) in enumerate(
    zip(top5_ids, top5_probs), start=1
):
    intent_name = muril_classifier.config.id2label[class_id.item()]

    print(
        f"{rank}. {intent_name} "
        f"→ {probability.item() * 100:.2f}%"
    )

Clean Hindi text: मेरे लिए ऑनलाइन सर्वेक्षणों के बारे में नवीनतम पॉडकास्ट चालू करें

Top 5 predicted intents:

1. play_podcasts → 97.00%
2. play_radio → 0.88%
3. play_audiobook → 0.60%
4. music_query → 0.28%
5. audio_volume_mute → 0.28%


In [ ]:
# ============================================================
# Cell 45: Show Real Hindi MASSIVE Podcast Utterances
# ============================================================

podcast_id = 58

count = 0

for example in train_data:
    if example["intent"] == podcast_id and example["locale"] == "hi-IN":
        print("Utterance:", example["utt"])
        count += 1

        if count == 10:
            break

print("\nShown:", count, "examples")

Utterance: वापस जाओ
Utterance: मेरे लिए ऑनलाइन सर्वेक्षणों के बारे में नवीनतम पॉडकास्ट चालू करें
Utterance: एक और कहानी से मेरे नवीनतम पॉडकास्ट चलाओ
Utterance: my favorite podcast चलाओ
Utterance: अगली रिकॉर्डिंग चलाओ
Utterance: अगले प्रकरण को छोड़ें
Utterance: अगला
Utterance: अगला कृपया
Utterance: छोड़ें
Utterance: इंडिया के स्कूलों में इंडिया के स्कूलों में debate पता लगाओ पता लगाओ

Shown: 10 examples


In [ ]:
print(type(trainer))
print(trainer.eval_dataset)

<class 'transformers.trainer.Trainer'>
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10165
})


In [ ]:
# ============================================================
# Cell 46: Evaluate Trained MuRIL Intent Classifier
# Purpose: Measure the model's performance on the held-out
#          evaluation/test dataset
# ============================================================

eval_results = trainer.evaluate()

print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.168971,0.802503,5,0.859911,0.802928,0.856656


Evaluation Results:
eval_loss: 0.8025031089782715
eval_accuracy: 0.8599114608952287
eval_macro_f1: 0.8029283298325255
eval_weighted_f1: 0.8566563356765532


In [ ]:
# Save model locally in Colab environment
trainer.save_model("./best_muril_indic_intent_model")
tokenizer.save_pretrained("./best_muril_indic_intent_model")

# (Optional) Mount Drive and save permanently
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./best_muril_indic_intent_model /content/drive/MyDrive/

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mounted at /content/drive
